# Requêtes sources du One Pager

Une section par KPI (numérotation de `docs/notes_et_questions.md`).

- **Requête retenue** : détail au grain fin (une ligne par ligne de production, par batch...), c'est elle qui alimentera les notebooks puis Power BI.
- **Contrôle** : donne la valeur du KPI à date du jour, uniquement pour vérifier la requête retenue.

Les KPI de déploiement D2 et D3 viennent uniquement du référentiel des activations de l'équipe DS (une ligne par ligne de production et par module). Nom de table provisoire : `mal_maite_common_prd.gold.ref_production_line_deployment`.

| KPI | Statut |
|---|---|
| D1 – Data digitalization | ✅ validé (14) |
| D2 – Metamodels | ⏳ en attente de la table DS |
| D3 – AI Recommendations | ⏳ en attente de la table DS |
| D4 – Time Tracking & Production Notes | ✅ validé (8) |
| Q1 – Uncaptured Aveva data | 🟡 en test |
| Q2 – Missing manual entries | 🟡 en test |
| Autres KPI | requêtes initiales, à reprendre une par une |

## D1 – Data digitalization ✅
Lignes ayant plus de 180 batchs non supprimés (tous statuts, cf. PO1). Date = `created_at` du 181e batch.

In [0]:
%sql
WITH batches_numerotes AS (
    SELECT
        pl.id_plant_production_line AS production_line_id,
        pl.common_code              AS production_line_code,
        pl.name                     AS production_line_name,
        b.id_batch,
        b.created_at,
        ROW_NUMBER() OVER (
            PARTITION BY pl.id_plant_production_line
            ORDER BY b.created_at, b.id_batch
        ) AS rang_batch
    FROM ext_mal_psql_maite_vision_board_prd.public.batches b
    INNER JOIN ext_mal_psql_maite_vision_board_prd.public.plants_production_lines pl
        ON pl.id_plant_production_line = b.production_line
    WHERE b.deleted = false
)
SELECT
    production_line_id,
    production_line_code,
    production_line_name,
    to_date(created_at) AS digitalization_date
FROM batches_numerotes
WHERE rang_batch = 181
ORDER BY digitalization_date;

In [0]:
%sql
SELECT COUNT(*) AS kpi_data_digitalization
FROM (
    SELECT pl.id_plant_production_line
    FROM ext_mal_psql_maite_vision_board_prd.public.batches b
    INNER JOIN ext_mal_psql_maite_vision_board_prd.public.plants_production_lines pl
        ON pl.id_plant_production_line = b.production_line
    WHERE b.deleted = false
    GROUP BY pl.id_plant_production_line
    HAVING COUNT(b.id_batch) > 180
) lignes_digitalisees;

## D2 – Metamodels ⏳
Lignes actives sur le module `metamodel` du référentiel DS. En attente de la livraison de la table DS.

In [0]:
%sql
SELECT
    production_line_id,
    production_line_code,
    site,
    is_active,
    valid_from,
    valid_to
FROM mal_maite_common_prd.gold.ref_production_line_deployment
WHERE module = 'metamodel'
ORDER BY valid_from;

In [0]:
%sql
SELECT COUNT(DISTINCT production_line_id) AS kpi_metamodels
FROM mal_maite_common_prd.gold.ref_production_line_deployment
WHERE module = 'metamodel'
  AND is_active = true;

## D3 – AI Recommendations ⏳
Lignes actives sur le module `recommendations` du référentiel DS. En attente de la livraison de la table DS.

In [0]:
%sql
SELECT
    production_line_id,
    production_line_code,
    site,
    is_active,
    valid_from,
    valid_to
FROM mal_maite_common_prd.gold.ref_production_line_deployment
WHERE module = 'recommendations'
ORDER BY valid_from;

In [0]:
%sql
SELECT COUNT(DISTINCT production_line_id) AS kpi_ai_recommendations
FROM mal_maite_common_prd.gold.ref_production_line_deployment
WHERE module = 'recommendations'
  AND is_active = true;

## D4 – Time Tracking & Production Notes ✅
Lignes ayant à la fois des notes de production (`batches_notes`, toutes) et une config TRS (`trs_metadata`, contenu tel quel). Aucun site exclu. Correspondance par nom de ligne en majuscules. Date = 1re note de production.

In [0]:
%sql
WITH lignes_trs AS (
    SELECT DISTINCT upper(trim(prd_line)) AS line_name
    FROM mal_maite_common_prd.gold.trs_metadata
),
notes_par_ligne AS (
    SELECT
        b.production_line   AS production_line_id,
        MIN(bn.created_at)  AS first_note_at,
        COUNT(*)            AS nb_notes
    FROM ext_mal_psql_maite_vision_board_prd.public.batches b
    INNER JOIN ext_mal_psql_maite_vision_board_prd.public.batches_notes bn
        ON bn.batch = b.id_batch
    WHERE b.deleted = false
    GROUP BY b.production_line
)
SELECT
    pl.id_plant_production_line AS production_line_id,
    pl.common_code              AS production_line_code,
    pl.name                     AS production_line_name,
    to_date(n.first_note_at)    AS deployment_date,
    n.nb_notes
FROM ext_mal_psql_maite_vision_board_prd.public.plants_production_lines pl
INNER JOIN notes_par_ligne n
    ON n.production_line_id = pl.id_plant_production_line
INNER JOIN lignes_trs t
    ON t.line_name = upper(trim(pl.name))
ORDER BY deployment_date;

In [0]:
%sql
SELECT DISTINCT md.prd_line
FROM mal_maite_common_prd.gold.trs_metadata md
LEFT JOIN ext_mal_psql_maite_vision_board_prd.public.plants_production_lines pl
    ON upper(trim(pl.name)) = upper(trim(md.prd_line))
WHERE pl.id_plant_production_line IS NULL;

## Q1 – Uncaptured Aveva data · Q2 – Missing manual entries 🟡
Même requête retenue pour les deux KPI (une ligne par recommandation). Le contrôle calcule Q1 avec les deux règles candidates (cf. PO4).

In [0]:
%sql
SELECT
    m.batch_id,
    m.id_parameter_localization,
    m.id_recommendation,
    m.id_production_line                          AS production_line_id,
    pl.common_code                                AS production_line_code,
    to_date(m.calculation_interval_after_datetime) AS expected_reco_date,
    m.flg_succes,
    m.flg_echec,
    m.has_auto_value,
    m.has_manual_value,
    m.id_alerting,
    m.type_echec
FROM mal_maite_bi_prd.recommendations_monitoring.fact_monitoring_auto_manual_entries m
LEFT JOIN ext_mal_psql_maite_vision_board_prd.public.plants_production_lines pl
    ON pl.id_plant_production_line = m.id_production_line
WHERE m.flg_succes = 1
   OR m.flg_echec = 1;

In [0]:
%sql
WITH periode AS (
    SELECT *
    FROM mal_maite_bi_prd.recommendations_monitoring.fact_monitoring_auto_manual_entries
    WHERE to_date(calculation_interval_after_datetime) >= date_sub(current_date(), 30)
      AND to_date(calculation_interval_after_datetime) <  current_date()
),
compteurs AS (
    SELECT
        SUM(COALESCE(flg_echec, 0) + COALESCE(flg_succes, 0))                                 AS recos_attendues,
        SUM(CASE WHEN has_auto_value = 1 AND has_manual_value = 0 THEN flg_echec ELSE 0 END)  AS echecs_auto_seuls,
        SUM(CASE WHEN id_alerting = 1 THEN flg_echec ELSE 0 END)                              AS echecs_chute_tags,
        SUM(CASE WHEN has_manual_value = 1 THEN flg_echec ELSE 0 END)                         AS echecs_manuels
    FROM periode
)
SELECT
    recos_attendues,
    echecs_auto_seuls,
    ROUND(100 * echecs_auto_seuls  / recos_attendues, 1) AS q1_pct_regle_dashboard,
    echecs_chute_tags,
    ROUND(100 * echecs_chute_tags  / recos_attendues, 1) AS q1_pct_regle_notebook,
    echecs_manuels,
    ROUND(100 * echecs_manuels     / recos_attendues, 1) AS q2_pct_missing_manual_entries
FROM compteurs;

## Requêtes initiales des autres KPI
A reprendre KPI par KPI (même méthode : requête retenue + contrôle).

In [0]:
%sql
------Maltimize unique users
SELECT * FROM ext_mal_psql_maite_vision_board_prd.public.profiles_users
WHERE deleted = 'false';

### U2 – Dashboards PowerBI unique users (requête initiale)
Pas encore de requête : source non disponible.